## Simple MNE example

In [1]:
import numpy as np
import mne
from mne.datasets import sample
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score



In [4]:
from pathlib import Path
from mne.datasets import sample
import mne

data_path = sample.data_path()

raw = mne.io.read_raw_fif(
    data_path / "MEG" / "sample" / "sample_audvis_raw.fif",
    preload=True
)

Opening raw data file C:\Users\Neermita\mne_data\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 166799  =      0.000 ...   277.714 secs...


In [5]:
raw

<Raw | sample_audvis_raw.fif, 376 x 166800 (277.7 s), ~481.7 MiB, data loaded>

In [6]:
# Select channels of interest (EEG channels)
picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False)

In [7]:
picks

array([315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327,
       328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340,
       341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353,
       354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366,
       368, 369, 370, 371, 372, 373, 374])

In [8]:
# Set the events and event_id
events = mne.find_events(raw, stim_channel='STI 014')
event_id = {'left/auditory': 1, 'right/auditory': 2}

Finding events on: STI 014
320 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]


In [9]:
# Create epochs around events
epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=0.5, picks=picks, baseline=(None, 0), preload=True)
labels = epochs.events[:, -1]

# Extract data and labels
X = epochs.get_data()
y = labels

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Not setting metadata
145 matching events found
Setting baseline interval to [-0.19979521315838786, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 145 events and 421 original time points ...
0 bad epochs dropped


In [10]:
labels

array([2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1,
       2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, 1, 2, 1, 2,
       1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1,
       2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2,
       1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, 1, 2, 1, 2,
       1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2,
       1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1])

In [16]:
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

# Initialize a classifier
svm = SVC(kernel='linear', C=1)

# Create a pipeline
clf = Pipeline([('CSP', csp), ('SVM', svm)])

# Train the classifier
clf.fit(X_train, y_train)

# Predict the labels for the test set
y_pred = clf.predict(X_test)

# Evaluate the classifier
print("Classification report:\n", classification_report(y_test, y_pred))
print("Accuracy score:", accuracy_score(y_test, y_pred))

Computing rank from data with rank=None


    Using tolerance 0.00029 (2.2e-16 eps * 59 dim * 2.2e+10  max singular value)
    Estimated rank (data): 59
    data: rank 59 computed from 59 data channels with 0 projectors
Reducing data rank from 59 -> 59
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Classification report:
               precision    recall  f1-score   support

           1       0.42      0.33      0.37        15
           2       0.41      0.50      0.45        14

    accuracy                           0.41        29
   macro avg       0.41      0.42      0.41        29
weighted avg       0.41      0.41      0.41        29

Accuracy score: 0.41379310344827586


## My data

In [1]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from mne.decoding import CSP
from sklearn.decomposition import PCA


In [4]:
# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
subject = "LTP063"
session = "0"
base_path = r"C:\Users\Neermita\Desktop\memory_and_task\ds004395"

sub_dir = f"sub-{subject}"
ses_dir = f"ses-{session}"

edf_path = os.path.join(base_path, sub_dir, ses_dir, "eeg", f"{sub_dir}_{ses_dir}_task-ltpFR_eeg.edf")
events_path = os.path.join(base_path, sub_dir, ses_dir, "eeg", f"{sub_dir}_{ses_dir}_task-ltpFR_events.tsv")
elec_path = os.path.join(base_path, sub_dir, ses_dir, "eeg", f"{sub_dir}_{ses_dir}_space-CapTrak_electrodes.tsv")

raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


In [5]:
raw

<RawEDF | sub-LTP063_ses-0_task-ltpFR_eeg.edf, 129 x 2603000 (5206.0 s), ~101 KiB, data not loaded>

In [6]:
# Montage Setup
electrodes = pd.read_csv(elec_path, sep='\t')
electrodes = electrodes[electrodes['x'] != 'n/a'].copy()
ch_pos = {row['name']: [float(row['x']), float(row['y']), float(row['z'])] 
          for _, row in electrodes.iterrows() if row['name'] in raw.ch_names}
raw.set_montage(mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head'), on_missing='ignore')

<RawEDF | sub-LTP063_ses-0_task-ltpFR_eeg.edf, 129 x 2603000 (5206.0 s), ~145 KiB, data not loaded>